# Règles de nettoyage — Périmètre commercial/industriel (DVF géolocalisé 2021-2025)

En miroir de `02_regles_nettoyage_dvf.ipynb` (résidentiel). Reprend l'investigation démarrée dans
`03_exploration_commercial.ipynb` (sections 1-8) pour aboutir à des règles de nettoyage définitives
sur le périmètre `type_local == 'Local industriel. commercial ou assimilé'`.

Fichier : `data/raw/dvf.csv` — même source, lue par chunks. Réutilise `keep_mask.npy` (dédoublonnage
global, calculé dans `02_regles_nettoyage_dvf.ipynb`) plutôt que de le recalculer.

In [52]:
import pandas as pd
import numpy as np
import os

CSV_PATH = r"C:\Users\eree\Documents\projet-immo-france\data\raw\dvf.csv"
CHUNK_SIZE = 1_000_000
TYPE_LOCAL_CIBLE = "Local industriel. commercial ou assimilé"

PROCESSED_DIR = os.path.join(os.path.dirname(os.path.dirname(CSV_PATH)), "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)
KEEP_MASK_PATH = os.path.join(PROCESSED_DIR, "keep_mask.npy")  # partagé avec le résidentiel

try:
    keep_mask
except NameError:
    keep_mask = np.load(KEEP_MASK_PATH)
    print(f"keep_mask rechargé depuis {KEEP_MASK_PATH}")

In [53]:
# Thème graphique commun — identique à 01/02/03
COLORS = {
    "bleu_clair": "#dbeafe",
    "bleu_secondaire": "#60a5fa",
    "bleu_primaire": "#2563eb",
    "bleu_marine": "#1e3a8a",
    "rouge_leger": "#fecaca",
    "rouge_modere": "#f87171",
    "rouge_exclusion": "#dc2626",
    "rouge_critique": "#7f1d1d",
}

import matplotlib.pyplot as plt
plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

## 9. Investigation des valeurs extrêmes de `valeur_fonciere` (>1 milliard €)

Même méthode que pour le cas parisien à 695M€ en résidentiel (notebook 02, Vérification 2bis) :
identifier les `id_mutation` concernés, vérifier si `valeur_fonciere` est identique sur toutes les
lignes d'un même `id_mutation` (signe de prix total réparti sur plusieurs lots), et regarder la
composition (surfaces, nombre de lignes) pour juger de la plausibilité.

In [54]:
cols_needed = ["id_mutation", "nature_mutation", "type_local", "valeur_fonciere",
               "surface_reelle_bati", "nombre_pieces_principales", "nombre_lots", "nom_commune"]

extremes_commercial = []
for chunk in pd.read_csv(CSV_PATH, usecols=cols_needed, chunksize=CHUNK_SIZE, low_memory=False):
    sub = chunk[
        (chunk["nature_mutation"] == "Vente") &
        (chunk["type_local"] == TYPE_LOCAL_CIBLE) &
        (chunk["valeur_fonciere"] > 1_000_000_000)
    ]
    if len(sub) > 0:
        extremes_commercial.append(sub)

df_extremes_commercial = pd.concat(extremes_commercial) if extremes_commercial else pd.DataFrame(columns=cols_needed)
print(f"Nombre de lignes avec valeur_fonciere > 1 milliard€ : {len(df_extremes_commercial):,}")
print(f"Nombre de id_mutation uniques parmi elles : {df_extremes_commercial['id_mutation'].nunique():,}")

Nombre de lignes avec valeur_fonciere > 1 milliard€ : 3
Nombre de id_mutation uniques parmi elles : 1


In [55]:
# Combien de lignes par id_mutation, et la valeur_fonciere est-elle identique sur chaque groupe ?
lignes_par_mutation_extremes = df_extremes_commercial.groupby("id_mutation").size().sort_values(ascending=False)
print(lignes_par_mutation_extremes.head(20))
print()

verif_identique = df_extremes_commercial.groupby("id_mutation")["valeur_fonciere"].nunique()
print("id_mutation où valeur_fonciere est identique sur toutes les lignes :",
      (verif_identique == 1).sum(), "/", len(verif_identique))
print()
print("Répartition : combien de id_mutation ont <=9 lignes (zone agrégation résidentielle) vs >9 (zone exclusion) ?")
print("<=9 lignes :", (lignes_par_mutation_extremes <= 9).sum())
print(">9 lignes  :", (lignes_par_mutation_extremes > 9).sum())

id_mutation
2021-175049    3
dtype: int64

id_mutation où valeur_fonciere est identique sur toutes les lignes : 1 / 1

Répartition : combien de id_mutation ont <=9 lignes (zone agrégation résidentielle) vs >9 (zone exclusion) ?
<=9 lignes : 1
>9 lignes  : 0


In [56]:
# Détail des 20 lignes les plus extrêmes, pour inspection au cas par cas
df_extremes_commercial.sort_values("valeur_fonciere", ascending=False)[
    ["id_mutation", "valeur_fonciere", "surface_reelle_bati", "nombre_pieces_principales", "nombre_lots", "nom_commune"]
].head(20)

,id_mutation,valeur_fonciere,surface_reelle_bati,nombre_pieces_principales,nombre_lots,nom_commune
490624,2021-175049,1.415000e+10,NaN,0.0,0,Marseille 8e Arrondissement
490625,2021-175049,1.415000e+10,3604.0,0.0,0,Marseille 8e Arrondissement
490626,2021-175049,1.415000e+10,720.0,0.0,0,Marseille 8e Arrondissement


### Interprétation — valeurs extrêmes (>1 milliard €)

**Résultat obtenu** : un seul cas dépasse 1 milliard € sur tout le périmètre commercial/industriel —
`id_mutation` 2021-175049, Marseille 8e Arrondissement, avec une `valeur_fonciere` de
**14 150 000 000€ (14.15 milliards)** répétée sur ses 3 lignes.

Composition :

| Ligne | surface_reelle_bati | nombre_pieces_principales | nombre_lots |
|---|---|---|---|
| 1 | NaN | 0 | 0 |
| 2 | 3 604 m² | 0 | 0 |
| 3 | 720 m² | 0 | 0 |

**Interprétation** : ce n'est **pas** le même phénomène que le cas parisien à 695M€ (résidentiel).
Là-bas, 18 lignes avec des surfaces réalistes et variées formaient un vrai immeuble cohérent. Ici :
- Seulement 3 lignes, avec des caractéristiques peu cohérentes entre elles (une surface manquante,
  deux surfaces très différentes 3 604 m² et 720 m²)
- `nombre_lots = 0` sur les 3 lignes — aucune structure de copropriété, incompatible avec un grand
  ensemble multi-unités
- Le prix au m² implicite (en sommant les 2 surfaces connues, 4 324 m²) donne **~3.27 millions
  €/m²** — un ordre de grandeur totalement hors marché, même pour l'immobilier le plus cher au monde
  (Monaco, Hong Kong tournent autour de 100 000-150 000€/m²)

**Conclusion** : tout pointe vers une **erreur de saisie** (probablement des zéros ajoutés par erreur
— 14 150 000€ serait un montant bien plus plausible pour ce type de bien) plutôt qu'une vraie
transaction, aussi extrême soit-elle.

**Point méthodologique important** : avec seulement 3 lignes (≤9), cette transaction tomberait dans
la **zone d'agrégation** si on réutilisait telle quelle la règle résidentielle
(`nb_lignes ≤ 9 → agréger`) — ce qui la garderait dans le dataset final avec un prix absurde. Ça
confirme ce que la section 5 avait déjà suggéré : le seuil de lignes résidentiel n'est **pas
suffisant à lui seul** pour ce périmètre. Il faut une règle complémentaire basée sur la
**plausibilité du prix** (ex : plafond absolu, ou ratio €/m² aberrant) plutôt que sur le seul nombre
de lignes — creusé dans les sections 10 à 13 ci-dessous.

## 10. Dédoublonnage strict — jamais vérifié sur ce périmètre

Toutes les sections précédentes (2 à 9) ont été calculées **sans retirer les doublons stricts au
préalable** — contrairement au résidentiel où le dédoublonnage (Vérification 4/5 du notebook 02) a
été fait avant toute analyse de composition. Le dédoublonnage résidentiel (`keep_mask.npy`) est basé
sur un hash de **toutes les colonnes**, calculé sur le fichier entier (20,4M lignes) — donc valable
pour n'importe quel périmètre, y compris commercial. Pas besoin de le recalculer : on le recharge
directement.

In [57]:
import numpy as np

try:
    keep_mask
except NameError:
    keep_mask = np.load(KEEP_MASK_PATH)  # même fichier que le résidentiel (dédoublonnage global)
    print(f"keep_mask rechargé depuis {KEEP_MASK_PATH}")

CHUNK_SIZE = 1_000_000
total_commercial_brut = 0
total_commercial_dedup = 0
global_idx = 0

for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE, low_memory=False, usecols=["type_local"]):
    n = len(chunk)
    chunk_keep_mask = keep_mask[global_idx:global_idx + n]
    global_idx += n

    is_commercial = (chunk["type_local"] == TYPE_LOCAL_CIBLE)
    total_commercial_brut += is_commercial.sum()
    total_commercial_dedup += (is_commercial & chunk_keep_mask).sum()

print(f"Lignes commercial/industriel avant dédoublonnage : {total_commercial_brut:,}")
print(f"Lignes commercial/industriel après dédoublonnage  : {total_commercial_dedup:,}")
print(f"Doublons stricts retirés : {total_commercial_brut - total_commercial_dedup:,} "
      f"({(total_commercial_brut - total_commercial_dedup) / total_commercial_brut * 100:.2f}%)")

Lignes commercial/industriel avant dédoublonnage : 650,255
Lignes commercial/industriel après dédoublonnage  : 632,166
Doublons stricts retirés : 18,089 (2.78%)


### Interprétation — dédoublonnage

**Résultat obtenu** :
- Avant dédoublonnage : 650 255 lignes
- Après dédoublonnage : 632 166 lignes
- Doublons stricts retirés : **18 089 (2.78%)**

**Interprétation** : le taux de duplication est nettement plus bas qu'en résidentiel (2.78% contre
11.45% sur Maison/Appartement, notebook 02). Le mécanisme de duplication semble bien réel sur ce
périmètre aussi, mais moins fréquent — cohérent avec l'idée que les transactions commerciales/
industrielles sont plus hétérogènes et moins standardisées qu'un lot de maisons individuelles, donc
statistiquement moins sujettes à un export dupliqué à l'identique. Pas d'action corrective au-delà du
filtrage déjà appliqué via `keep_mask` — juste un chiffre à retenir pour la documentation finale.

## 11. Vente en bloc systématique (méthode adaptée au périmètre commercial)

**Différence importante avec le résidentiel** : la Vérification 5 du notebook 02 utilisait
l'homogénéité de `type_local` (Maison vs Appartement) comme signal discriminant. Ici, ça ne
fonctionne pas — toutes les lignes de ce périmètre partagent par construction le même `type_local`
(`Local industriel. commercial ou assimilé`), donc l'homogénéité serait toujours 100%, sans aucune
valeur informative.

**Signal de remplacement retenu** : la variabilité de `surface_reelle_bati` *au sein* d'une même
transaction (coefficient de variation = écart-type / moyenne). Intuition : une propriété avec
annexes (entrepôt principal + petit local technique) devrait avoir des surfaces du même ordre de
grandeur ; un vrai ensemble immobilier hétérogène (mélange de grandes et petites unités) devrait
avoir une variabilité plus forte. À vérifier — hypothèse non confirmée, contrairement à
l'homogénéité `type_local` en résidentiel qui, elle, s'appuyait sur une distinction déjà connue et
validée.

In [58]:
cols_needed = ["nature_mutation", "type_local", "id_mutation", "valeur_fonciere", "surface_reelle_bati"]

try:
    keep_mask
except NameError:
    keep_mask = np.load(KEEP_MASK_PATH)

resume_commercial_vb = []
global_idx = 0

for chunk in pd.read_csv(CSV_PATH, usecols=cols_needed, chunksize=CHUNK_SIZE, low_memory=False):
    n = len(chunk)
    chunk_keep_mask = keep_mask[global_idx:global_idx + n]
    chunk_dedup = chunk[chunk_keep_mask]
    global_idx += n

    subset = chunk_dedup[
        (chunk_dedup["nature_mutation"] == "Vente") &
        (chunk_dedup["type_local"] == TYPE_LOCAL_CIBLE)
    ]
    if len(subset) > 0:
        g = subset.groupby("id_mutation").agg(
            nb_lignes=("valeur_fonciere", "size"),
            valeur_min=("valeur_fonciere", "min"),
            valeur_max=("valeur_fonciere", "max"),
            surface_totale=("surface_reelle_bati", "sum"),
            surface_std=("surface_reelle_bati", "std"),
            surface_moyenne=("surface_reelle_bati", "mean"),
        )
        resume_commercial_vb.append(g)

all_commercial_vb = pd.concat(resume_commercial_vb)
final_commercial_vb = all_commercial_vb.groupby("id_mutation").agg(
    nb_lignes=("nb_lignes", "sum"),
    valeur_min=("valeur_min", "min"),
    valeur_max=("valeur_max", "max"),
    surface_totale=("surface_totale", "sum"),
    surface_std=("surface_std", "mean"),       # approximation : moyenne des std partielles par bloc
    surface_moyenne=("surface_moyenne", "mean"),
)

final_commercial_vb["valeur_identique"] = final_commercial_vb["valeur_min"] == final_commercial_vb["valeur_max"]
candidats_vb_commercial = final_commercial_vb[
    (final_commercial_vb["nb_lignes"] > 1) & (final_commercial_vb["valeur_identique"])
].copy()

print(f"Transactions multi-lignes, valeur identique (candidats vente en bloc) : {len(candidats_vb_commercial):,}")
print(f"Sur un total de {len(final_commercial_vb):,} transactions commerciales dédupliquées")

Transactions multi-lignes, valeur identique (candidats vente en bloc) : 90,045
Sur un total de 393,103 transactions commerciales dédupliquées


In [59]:
candidats_vb_commercial["cv_surface"] = (
    candidats_vb_commercial["surface_std"] / candidats_vb_commercial["surface_moyenne"]
)

bins = [2, 3, 5, 10, 20, 50, 100, candidats_vb_commercial["nb_lignes"].max() + 1]
labels = ["2", "3-4", "5-9", "10-19", "20-49", "50-99", "100+"]
candidats_vb_commercial["tranche"] = pd.cut(
    candidats_vb_commercial["nb_lignes"], bins=bins, labels=labels, right=False
)

print("Nombre de transactions par tranche :")
print(candidats_vb_commercial["tranche"].value_counts().sort_index())
print()
print("Coefficient de variation médian de la surface, par tranche :")
print(candidats_vb_commercial.groupby("tranche", observed=True)["cv_surface"].median())

Nombre de transactions par tranche :
tranche
2        60056
3-4      19571
5-9       7493
10-19     2091
20-49      664
50-99      120
100+        50
Name: count, dtype: int64

Coefficient de variation médian de la surface, par tranche :
tranche
2        0.317477
3-4      0.630765
5-9      0.883077
10-19    1.287512
20-49    1.584560
50-99    2.009289
100+     2.525216
Name: cv_surface, dtype: float64


### Interprétation — vente en bloc, périmètre commercial

**Résultat obtenu** : 90 045 transactions candidates ("vente en bloc" : multi-lignes + `valeur_fonciere`
identique) sur 393 103 transactions commerciales dédupliquées, soit **22.9%** — une proportion
nettement plus élevée qu'en résidentiel (14.7% des transactions résiduelles, Vérification 5 du
notebook 02).

| Tranche | Volume | % | CV surface médian |
|---|---|---|---|
| 2 | 60 056 | 66.7% | 0.317 |
| 3-4 | 19 571 | 21.7% | 0.631 |
| 5-9 | 7 493 | 8.3% | 0.883 |
| 10-19 | 2 091 | 2.3% | 1.288 |
| 20-49 | 664 | 0.7% | 1.585 |
| 50-99 | 120 | 0.1% | 2.009 |
| 100+ | 50 | 0.06% | 2.525 |

**Interprétation** :
- La concentration en volume ressemble au résidentiel : **96.7%** des candidats (87 120 sur 90 045)
  ont 9 lignes ou moins.
- Contrairement au résidentiel (où l'homogénéité `type_local` baissait progressivement sans vraie
  rupture), ici le **coefficient de variation de la surface augmente de façon quasi monotone et
  marquée** avec le nombre de lignes : de 0.32 (2 lignes) à 2.52 (100+ lignes), soit une
  multiplication par ~8. C'est un signal plus net que celui du résidentiel, même s'il reste progressif
  (pas de cassure franche non plus).
- Cohérent avec l'intuition : les petites transactions (2-4 lignes) ont des surfaces relativement
  homogènes entre elles (CV<0.65 — probablement plusieurs petits locaux similaires ou une propriété +
  annexe), alors que les grosses transactions (50+ lignes) mélangent des surfaces très disparates
  (CV>2 — cohérent avec un vrai ensemble immobilier hétérogène : commerces + entrepôts + bureaux dans
  un même programme).

**⚠️ Point méthodologique à vérifier avant de conclure** : ce calcul suppose que `surface_reelle_bati`
est bien une valeur **par ligne**, comme pour le résidentiel. Or la section 13 (juste après) révèle
des cas où la **même** surface est répétée à l'identique sur plusieurs lignes d'un même `id_mutation`
— exactement le même mécanisme que `valeur_fonciere` pour les ventes en bloc. Si ce phénomène est
fréquent, `surface_totale` (calculé par somme) et donc `cv_surface` pourraient être faussés pour une
partie des transactions. À corriger avant de figer un seuil définitif (cf. section 13).

## 12. Règle de plausibilité de prix — ratio €/m²

Objectif : construire le garde-fou évoqué après le cas Marseille (14.15 milliards €, 3.27M€/m²
implicite) — un seuil basé sur le prix au m², pas seulement sur le nombre de lignes, pour attraper
les erreurs de saisie que la règle "vente en bloc" seule laisserait passer.

In [60]:
cols_needed = ["nature_mutation", "type_local", "valeur_fonciere", "surface_reelle_bati"]

ratio_prix_m2 = pd.Series(dtype="float64")

for chunk in pd.read_csv(CSV_PATH, usecols=cols_needed, chunksize=CHUNK_SIZE, low_memory=False):
    sub = chunk[
        (chunk["nature_mutation"] == "Vente") &
        (chunk["type_local"] == TYPE_LOCAL_CIBLE) &
        (chunk["surface_reelle_bati"] > 0) &
        (chunk["valeur_fonciere"].notna())
    ]
    ratio = sub["valeur_fonciere"] / sub["surface_reelle_bati"]
    ratio_prix_m2 = pd.concat([ratio_prix_m2, ratio])

print(f"Lignes avec ratio €/m² calculable : {len(ratio_prix_m2):,}")
ratio_prix_m2.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.9, 0.99, 0.999, 0.9999])

Lignes avec ratio €/m² calculable : 579,627


count     5.796270e+05
mean      6.407691e+04
std       7.485831e+05
min       1.075142e-05
1%        2.000000e-02
25%       9.534368e+02
50%       2.616822e+03
75%       9.047619e+03
90%       4.700000e+04
99%       1.095391e+06
99.9%     6.440000e+06
99.99%    2.898000e+07
max       8.660147e+07
dtype: float64

### Interprétation — plausibilité €/m²

**Résultat obtenu** (579 627 lignes avec ratio calculable) :

| Percentile | €/m² |
|---|---|
| Min | 0.00001€ |
| 1% | 0.02€ |
| 25% | 953€ |
| 50% (médiane) | 2 617€ |
| 75% | 9 048€ |
| 90% | 47 000€ |
| 99% | 1 095 391€ |
| 99.9% | 6 440 000€ |
| 99.99% | 28 980 000€ |
| **Max** | **86 601 470€** |

**Interprétation** :
- La médiane (2 617€/m²) est un ordre de grandeur plausible pour un local commercial standard.
- La queue haute est vertigineuse : le maximum (86.6M€/m²) dépasse même les ratios implicites du cas
  Marseille (~3.9M€/m² et ~19.7M€/m² selon la ligne, en excluant la ligne à surface manquante) — il
  existe donc au moins un cas encore plus extrême ailleurs dans le dataset, jamais examiné en détail.
- Les deux lignes exploitables du cas Marseille se situent respectivement autour du 99e-99.9e
  percentile (~3.9M€) et juste sous le 99.99e (~19.7M€) — cohérent avec l'hypothèse d'erreur de
  saisie : ce ne sont pas des valeurs "un peu hautes", elles sont dans l'extrême queue de la
  distribution.
- **Le bas de la distribution mérite aussi un contrôle** : des ratios proches de 0€/m² (0.00001€,
  0.02€) sont tout aussi suspects que les valeurs extrêmes hautes — soit de vraies transactions
  symboliques (cession à l'euro symbolique), soit des erreurs de saisie sur `valeur_fonciere` ou
  `surface_reelle_bati`.

**Proposition de règle (à valider)** : un plafond autour du 99.9e-99.99e percentile (6.44M€/m² à
29M€/m²) capturerait le cas Marseille et les cas similaires, sans exclure à tort de vraies
transactions de bureaux/commerces premium en centre-ville dense.

## 13. Vérification du maximum de surface (646 230 m²)

Dernier point ouvert : le maximum de `surface_reelle_bati` (§7) est-il un vrai grand site logistique,
ou une autre erreur de saisie du même type que le cas Marseille ?

In [61]:
cols_needed = ["id_mutation", "nature_mutation", "type_local", "valeur_fonciere",
               "surface_reelle_bati", "nombre_pieces_principales", "nom_commune"]

top_surfaces = []
for chunk in pd.read_csv(CSV_PATH, usecols=cols_needed, chunksize=CHUNK_SIZE, low_memory=False):
    sub = chunk[chunk["type_local"] == TYPE_LOCAL_CIBLE]
    top_surfaces.append(sub.nlargest(20, "surface_reelle_bati"))

df_top_surfaces = pd.concat(top_surfaces).nlargest(20, "surface_reelle_bati")
df_top_surfaces["prix_m2"] = df_top_surfaces["valeur_fonciere"] / df_top_surfaces["surface_reelle_bati"]
df_top_surfaces[["id_mutation", "valeur_fonciere", "surface_reelle_bati", "prix_m2", "nom_commune"]]

,id_mutation,valeur_fonciere,surface_reelle_bati,prix_m2,nom_commune
17872062,2025-407957,243596.0,646230.0,0.376949,Grayan-et-l'Hôpital
17872149,2025-407957,243596.0,646230.0,0.376949,Grayan-et-l'Hôpital
17872227,2025-407957,243596.0,646230.0,0.376949,Grayan-et-l'Hôpital
17872305,2025-407957,243596.0,646230.0,0.376949,Grayan-et-l'Hôpital
17872440,2025-407957,243596.0,646230.0,0.376949,Grayan-et-l'Hôpital
17872476,2025-407957,243596.0,646230.0,0.376949,Grayan-et-l'Hôpital
17872570,2025-407957,243596.0,646230.0,0.376949,Grayan-et-l'Hôpital
17872602,2025-407957,243596.0,646230.0,0.376949,Grayan-et-l'Hôpital
15359446,2024-755990,NaN,593400.0,NaN,Calais
4165958,2021-1506457,NaN,450000.0,NaN,Saint-Léger-la-Montagne


### Interprétation — maximum de surface

**Résultat obtenu** (top 20 par surface) — plusieurs profils distincts apparaissent :

1. **Cas suspect n°1 (le maximum, 646 230 m²)** : `id_mutation` 2025-407957, Grayan-et-l'Hôpital —
   **8 lignes**, toutes avec **exactement la même surface** (646 230 m²) et le même prix (243 596€,
   soit 0.38€/m²). Une même surface répétée à l'identique sur 8 lignes ressemble au même mécanisme
   que `valeur_fonciere` pour les ventes en bloc : la surface totale du bâtiment semble enregistrée
   une fois par lot plutôt que répartie — donc probablement **pas 8 bâtiments de 646 230 m² chacun**
   (5.2 millions de m² cumulés serait absurde pour cette commune), mais un seul bâtiment de
   646 230 m², sur-compté si on somme naïvement par transaction.
2. **Même mécanisme suspecté** sur `id_mutation` 2024-730864 (Ressons-sur-Matz, 3 lignes, surface et
   prix identiques à chaque fois) et `id_mutation` 2021-698892 (Moliets-et-Maa, 2 lignes identiques).
3. **Cas probablement légitime** : `id_mutation` 2024-2425 (Loyettes), 265 000 m² pour 27.5M€, soit
   **103.77€/m²** — un ratio tout à fait plausible pour un grand site industriel/logistique. Pas
   d'alerte ici.
4. **Cas à `valeur_fonciere` manquante** : plusieurs des plus grandes surfaces (Calais,
   Saint-Léger-la-Montagne, Saint-Sylvestre...) n'ont **aucun prix renseigné** — pas d'alerte de
   plausibilité possible via le ratio €/m² sur ces lignes, mais ce sont aussi des lignes à exclure ou
   traiter à part faute de prix exploitable pour le modèle.

**Conclusion** : le maximum de 646 230 m² n'est probablement **pas** un vrai site unique de cette
taille, mais un artefact du même mécanisme de répétition déjà vu sur `valeur_fonciere` — appliqué
cette fois à `surface_reelle_bati`. **Ça confirme et renforce l'alerte méthodologique de la section
11** : avant de sommer les surfaces par transaction pour calculer `cv_surface`, il faut d'abord
vérifier si la surface est dupliquée ou réellement différente par ligne, sous peine de fausser à la
fois la détection "vente en bloc" et toute future variable dérivée (prix au m², surface totale du
bien).

## 14. Correction — détection de la surface dupliquée + recalcul du CV

Suite à la découverte de la section 13 : avant de faire confiance au `cv_surface` de la section 11,
on vérifie, pour chaque transaction candidate "vente en bloc", si `surface_reelle_bati` est répétée
à l'identique sur toutes ses lignes (comme `valeur_fonciere`) — auquel cas il faut compter la surface
**une seule fois**, pas la sommer ni calculer un écart-type dessus.

In [62]:
cols_needed = ["nature_mutation", "type_local", "id_mutation", "valeur_fonciere", "surface_reelle_bati"]

try:
    keep_mask
except NameError:
    keep_mask = np.load(KEEP_MASK_PATH)

resume_surfaces = []
global_idx = 0

for chunk in pd.read_csv(CSV_PATH, usecols=cols_needed, chunksize=CHUNK_SIZE, low_memory=False):
    n = len(chunk)
    chunk_keep_mask = keep_mask[global_idx:global_idx + n]
    chunk_dedup = chunk[chunk_keep_mask]
    global_idx += n

    subset = chunk_dedup[
        (chunk_dedup["nature_mutation"] == "Vente") &
        (chunk_dedup["type_local"] == TYPE_LOCAL_CIBLE)
    ]
    if len(subset) > 0:
        g = subset.groupby("id_mutation").agg(
            nb_lignes=("valeur_fonciere", "size"),
            valeur_min=("valeur_fonciere", "min"),
            valeur_max=("valeur_fonciere", "max"),
            surfaces=("surface_reelle_bati", lambda x: tuple(x.dropna())),
        )
        resume_surfaces.append(g)

all_surfaces = pd.concat(resume_surfaces)
final_surfaces = all_surfaces.groupby("id_mutation").agg(
    nb_lignes=("nb_lignes", "sum"),
    valeur_min=("valeur_min", "min"),
    valeur_max=("valeur_max", "max"),
    surfaces=("surfaces", "sum"),  # concatène les tuples entre chunks (un même id_mutation peut être coupé en 2 blocs)
)

final_surfaces["valeur_identique"] = final_surfaces["valeur_min"] == final_surfaces["valeur_max"]
candidats_correction = final_surfaces[
    (final_surfaces["nb_lignes"] > 1) & (final_surfaces["valeur_identique"])
].copy()

candidats_correction["nb_surfaces_renseignees"] = candidats_correction["surfaces"].apply(len)
candidats_correction["nb_surfaces_distinctes"] = candidats_correction["surfaces"].apply(lambda s: len(set(s)))
candidats_correction["surface_dupliquee"] = (
    (candidats_correction["nb_surfaces_renseignees"] > 1) &
    (candidats_correction["nb_surfaces_distinctes"] == 1)
)

print(f"Transactions candidates vente en bloc : {len(candidats_correction):,}")
print(f"Dont surface identique répétée sur toutes les lignes renseignées : "
      f"{candidats_correction['surface_dupliquee'].sum():,} "
      f"({candidats_correction['surface_dupliquee'].mean() * 100:.1f}%)")

Transactions candidates vente en bloc : 90,045
Dont surface identique répétée sur toutes les lignes renseignées : 18,350 (20.4%)


In [63]:
def surface_totale_corrigee(row):
    if len(row["surfaces"]) == 0:
        return np.nan
    if row["surface_dupliquee"]:
        return row["surfaces"][0]  # une seule vraie valeur, pas la somme des répétitions
    return sum(row["surfaces"])

def cv_corrige(row):
    if len(row["surfaces"]) <= 1:
        return np.nan
    if row["surface_dupliquee"]:
        return 0.0  # une seule vraie valeur -> pas de variation réelle à mesurer
    arr = np.array(row["surfaces"])
    return arr.std() / arr.mean() if arr.mean() > 0 else np.nan

candidats_correction["surface_totale_corrigee"] = candidats_correction.apply(surface_totale_corrigee, axis=1)
candidats_correction["cv_surface_corrige"] = candidats_correction.apply(cv_corrige, axis=1)

bins = [2, 3, 5, 10, 20, 50, 100, candidats_correction["nb_lignes"].max() + 1]
labels = ["2", "3-4", "5-9", "10-19", "20-49", "50-99", "100+"]
candidats_correction["tranche"] = pd.cut(candidats_correction["nb_lignes"], bins=bins, labels=labels, right=False)

print("Part de transactions à surface dupliquée, par tranche :")
print((candidats_correction.groupby("tranche", observed=True)["surface_dupliquee"].mean() * 100).round(1))
print()
print("CV surface — AVANT correction (section 11) vs APRÈS correction (surface dupliquée traitée comme valeur unique) :")
comparatif = pd.DataFrame({
    "cv_avant_correction": [0.317477, 0.630765, 0.883077, 1.287512, 1.584560, 2.009289, 2.525216],
    "cv_apres_correction": candidats_correction.groupby("tranche", observed=True)["cv_surface_corrige"].median().values,
}, index=labels)
comparatif

Part de transactions à surface dupliquée, par tranche :
tranche
2        26.1
3-4      11.0
5-9       5.3
10-19     4.1
20-49     4.7
50-99     7.5
100+      8.0
Name: surface_dupliquee, dtype: float64

CV surface — AVANT correction (section 11) vs APRÈS correction (surface dupliquée traitée comme valeur unique) :


,cv_avant_correction,cv_apres_correction
2,0.317477,0.224490
3-4,0.630765,0.523447
5-9,0.883077,0.802825
10-19,1.287512,1.229946
20-49,1.584560,1.560938
50-99,2.009289,1.993773
100+,2.525216,2.519076


### Interprétation — impact de la correction

**Résultat obtenu** :

| Tranche | % surface dupliquée | CV avant correction | CV après correction |
|---|---|---|---|
| 2 | 26.1% | 0.317 | 0.224 |
| 3-4 | 11.0% | 0.631 | 0.523 |
| 5-9 | 5.3% | 0.883 | 0.803 |
| 10-19 | 4.1% | 1.288 | 1.230 |
| 20-49 | 4.7% | 1.585 | 1.561 |
| 50-99 | 7.5% | 2.009 | 1.994 |
| 100+ | 8.0% | 2.525 | 2.519 |

**Interprétation** :
- La surface dupliquée est en fait **plus fréquente sur les petites transactions** (26.1% à 2
  lignes) que sur les grosses (4-8%) — l'inverse de l'hypothèse initiale. Ça nuance l'analogie faite
  avec le résidentiel pour le cas à 2 lignes : une partie de ces transactions ne sont probablement pas
  des "propriété + annexe" (surfaces différentes, cohérent avec le résidentiel) mais plutôt "un même
  bâtiment scindé en 2 lots administratifs" (prix ET surface du bâtiment entier collés sur chaque
  lot).
- **La conclusion qualitative de la section 11 tient malgré tout** : le CV continue d'augmenter de
  façon quasi monotone après correction (0.224 → 2.519). L'effet de la correction est réel mais
  modeste, concentré sur les tranches où la duplication est la plus fréquente (2-4 lignes) — le seuil
  `nb_lignes ≤ 9` envisagé reste défendable, la correction ne change pas la décision, elle l'affine.

**⚠️ Conséquence pour la section 12 (plausibilité €/m²)** : le ratio implicite de la ligne Marseille
à 3 604 m² (~3.93M€/m²) se situe *entre* le 99e percentile (1.1M€/m²) et le 99.9e (6.44M€/m²). Un
plafond fixé à 6.44M€/m² (comme envisagé en section 12) **ne l'attraperait pas** — un seuil plus bas,
plutôt autour du 99e percentile, serait nécessaire pour capturer ce cas précis. À trancher avant de
figer la règle de plausibilité de prix dans le pipeline complet.

## 15. Pipeline complet — application de toutes les règles et volumétrie finale

Applique **toutes** les règles tranchées jusqu'ici (sections 9 à 14) en une seule passe :
dédoublonnage, filtre Vente + VEFA, périmètre commercial/industriel, prix connu, géographie (règle
reprise telle quelle du résidentiel — contrôle générique sur longitude/latitude, indépendant du type
de bien), règle vente en bloc (`nb_lignes ≤ 9` → agréger avec correction de surface dupliquée,
`> 9` → exclure), puis seuil de plausibilité €/m².

In [64]:
CHUNK_SIZE = 1_000_000
cols_needed = ["nature_mutation", "type_local", "id_mutation", "valeur_fonciere",
               "surface_reelle_bati", "longitude", "latitude", "code_departement"]

NATURES_A_GARDER = {"Vente", "Vente en l'état futur d'achèvement"}
DOM_CODES = {"971", "972", "973", "974", "975", "976"}
LON_MIN, LON_MAX = -5.5, 10.0
LAT_MIN, LAT_MAX = 41.0, 51.5
SEUIL_VENTE_EN_BLOC = 9  # même seuil que le résidentiel, cf. sections 11/14

try:
    keep_mask
except NameError:
    keep_mask = np.load(KEEP_MASK_PATH)

lignes_finales_commercial = []
global_idx = 0

for chunk in pd.read_csv(CSV_PATH, usecols=cols_needed, chunksize=CHUNK_SIZE, low_memory=False):
    n = len(chunk)
    chunk_keep_mask = keep_mask[global_idx:global_idx + n]
    global_idx += n
    chunk_dedup = chunk[chunk_keep_mask]  # dédoublonnage strict, section 10

    perimetre = chunk_dedup[
        chunk_dedup["nature_mutation"].isin(NATURES_A_GARDER) &          # Vente + VEFA
        (chunk_dedup["type_local"] == TYPE_LOCAL_CIBLE) &                # périmètre commercial/industriel
        chunk_dedup["valeur_fonciere"].notna()                           # prix exploitable
    ].copy()

    # Géographie : règle reprise telle quelle du résidentiel (contrôle générique lon/lat, pas
    # revérifiée spécifiquement sur ce périmètre)
    is_dom = perimetre["code_departement"].astype(str).isin(DOM_CODES)
    coord_ok = (
        perimetre["longitude"].notna() & perimetre["latitude"].notna() &
        (is_dom | (perimetre["longitude"].between(LON_MIN, LON_MAX) & perimetre["latitude"].between(LAT_MIN, LAT_MAX)))
    )
    perimetre = perimetre[coord_ok]

    if len(perimetre) > 0:
        lignes_finales_commercial.append(perimetre[["id_mutation", "valeur_fonciere", "surface_reelle_bati"]])

df_final_lignes_commercial = pd.concat(lignes_finales_commercial)
print(f"Lignes après dédoublonnage, Vente+VEFA, périmètre commercial, prix connu, géo valide : "
      f"{len(df_final_lignes_commercial):,}")

# Règle vente en bloc (sections 11/14) : regrouper par id_mutation, avec correction de surface dupliquée
g = df_final_lignes_commercial.groupby("id_mutation").agg(
    nb_lignes=("valeur_fonciere", "size"),
    valeur_min=("valeur_fonciere", "min"),
    valeur_max=("valeur_fonciere", "max"),
    surfaces=("surface_reelle_bati", lambda x: tuple(x.dropna())),
)

def surface_corrigee(surfaces):
    if len(surfaces) == 0:
        return np.nan
    if len(set(surfaces)) == 1:          # surface dupliquée à l'identique -> une seule vraie valeur (section 14)
        return surfaces[0]
    return sum(surfaces)                 # surfaces distinctes -> vraie somme (plusieurs bâtiments)

g["surface_totale"] = g["surfaces"].apply(surface_corrigee)
g["valeur_fonciere"] = g["valeur_max"]   # valeur unique (identique sur les candidats vente en bloc ;
                                          # pour les rares cas "valeur différente", reprend le max par défaut,
                                          # comme le résidentiel où ce cas s'est révélé être un artefact sans impact

transactions_finales_commercial = g[g["nb_lignes"] <= SEUIL_VENTE_EN_BLOC].reset_index()
nb_exclues_vb = len(g) - len(transactions_finales_commercial)

print(f"Transactions exclues comme vraie vente en bloc (nb_lignes > {SEUIL_VENTE_EN_BLOC}) : {nb_exclues_vb:,}")
print(f"Transactions finales (avant seuil de plausibilité €/m²) : {len(transactions_finales_commercial):,}")

Lignes après dédoublonnage, Vente+VEFA, périmètre commercial, prix connu, géo valide : 601,520
Transactions exclues comme vraie vente en bloc (nb_lignes > 9) : 2,999
Transactions finales (avant seuil de plausibilité €/m²) : 391,232


### Interprétation — pipeline complet

**Résultat obtenu** :
- Lignes après dédoublonnage + Vente/VEFA + périmètre commercial + prix connu + géo valide : **601 520**
- Transactions exclues comme vraie vente en bloc (nb_lignes > 9) : **2 999**
- Transactions finales (avant seuil €/m²) : **391 232**

**Interprétation** : cohérent avec les chiffres déjà établis. Le périmètre commercial dédupliqué et
filtré Vente+VEFA (625 797 lignes, cf. section 17) perd encore ~3.9% (24 277 lignes) sur le contrôle
prix connu + géographie, avant de se regrouper en 391 232 + 2 999 = 394 231 transactions distinctes.

Les 2 999 exclusions "vraie vente en bloc" représentent une part bien plus faible que le total des
candidats trouvés en section 11 (90 045, toutes tranches confondues) — logique, puisque seule la queue
à >9 lignes (tranches 10-19 à 100+ : 2 091+664+120+50 = 2 925, très proche des 2 999 observés ici) est
réellement exclue. Le reste (≤9 lignes, 87 120 candidats) est agrégé et reste dans les transactions
finales.

## 16. Application du seuil de plausibilité €/m² — compte définitif

Dernière étape : exclure les transactions dont le ratio €/m² dépasse le seuil retenu (section 12/14
— environ le 99e percentile observé, 1 095 391€/m², plutôt que le 99.9e envisagé initialement, pour
bien capturer des cas comme Marseille). Les transactions sans surface exploitable ne peuvent pas être
vérifiées par ce ratio et sont conservées telles quelles.

In [65]:
SEUIL_PRIX_M2_MAX = 1_100_000  # ~99e percentile observé en section 12 (1 095 391€/m²) — à valider/ajuster

surface_connue = transactions_finales_commercial["surface_totale"] > 0
ratio_prix_m2 = transactions_finales_commercial["valeur_fonciere"] / transactions_finales_commercial["surface_totale"]
depasse_seuil = surface_connue & (ratio_prix_m2 > SEUIL_PRIX_M2_MAX)

transactions_vraiment_finales_commercial = transactions_finales_commercial[~depasse_seuil].copy()
nb_exclues_prix = depasse_seuil.sum()

print(f"Transactions exclues par le seuil de plausibilité €/m² (>{SEUIL_PRIX_M2_MAX:,}€/m²) : {nb_exclues_prix:,}")
print(f"Dont sans surface exploitable (non vérifiables, conservées) : {(~surface_connue).sum():,}")
print()
print(f"COMPTE DÉFINITIF DU PÉRIMÈTRE COMMERCIAL/INDUSTRIEL NETTOYÉ : {len(transactions_vraiment_finales_commercial):,}")

Transactions exclues par le seuil de plausibilité €/m² (>1,100,000€/m²) : 44
Dont sans surface exploitable (non vérifiables, conservées) : 18,164

COMPTE DÉFINITIF DU PÉRIMÈTRE COMMERCIAL/INDUSTRIEL NETTOYÉ : 391,188


### Interprétation — compte définitif

**Résultat obtenu** :
- Transactions exclues par le seuil €/m² (>1 100 000€/m²) : **44**
- Transactions sans surface exploitable (non vérifiables, conservées) : **18 164**
- **COMPTE DÉFINITIF : 391 188 transactions**

**Interprétation** : le seuil de plausibilité ne retire que 44 transactions (0.011% du volume à cette
étape) — un impact minime en volume, mais qui inclut très probablement le cas Marseille
(`id_mutation` 2021-175049) : avec un prix de 14.15 milliards€ et une surface totale corrigée de
4 324 m² (3 604 + 720, la ligne à surface manquante étant ignorée), son ratio implicite
(~3.27M€/m²) dépasse largement le seuil de 1.1M€/m².

Le nombre de transactions sans surface exploitable (18 164, soit 4.6% du volume à cette étape) est
cohérent avec le taux de valeurs manquantes de `surface_reelle_bati` observé en section 3 (5.54% au
niveau ligne) — ces transactions ne bénéficient pas du contrôle de plausibilité mais restent dans le
dataset final, faute de moyen de les vérifier.

**COMPTE DÉFINITIF DU PÉRIMÈTRE COMMERCIAL/INDUSTRIEL NETTOYÉ : 391 188 transactions.**

## 17. Tableau entonnoir — volumétrie cumulée à chaque étape

Récapitulatif du nombre de lignes/transactions restantes après chaque règle de nettoyage, dans l'ordre
où elles s'appliqueraient dans le job Talend — même logique que le tableau entonnoir du notebook 02.

In [66]:
try:
    keep_mask
except NameError:
    keep_mask = np.load(KEEP_MASK_PATH)

CHUNK_SIZE = 1_000_000
cols_needed = ["nature_mutation", "type_local", "valeur_fonciere"]

NATURES_A_GARDER = {"Vente", "Vente en l'état futur d'achèvement"}

funnel_commercial = {
    "1_total_lignes_brutes": 0,
    "2_apres_dedoublonnage_strict": 0,
    "3_vente_ou_vefa_tout_type": 0,
    "4_perimetre_commercial": 0,
}

global_idx = 0
for chunk in pd.read_csv(CSV_PATH, usecols=cols_needed, chunksize=CHUNK_SIZE, low_memory=False):
    n = len(chunk)
    funnel_commercial["1_total_lignes_brutes"] += n

    chunk_keep_mask = keep_mask[global_idx:global_idx + n]
    global_idx += n
    chunk_dedup = chunk[chunk_keep_mask]
    funnel_commercial["2_apres_dedoublonnage_strict"] += len(chunk_dedup)

    ventes = chunk_dedup[chunk_dedup["nature_mutation"].isin(NATURES_A_GARDER)]
    funnel_commercial["3_vente_ou_vefa_tout_type"] += len(ventes)

    commercial = ventes[ventes["type_local"] == TYPE_LOCAL_CIBLE]
    funnel_commercial["4_perimetre_commercial"] += len(commercial)

print(f"{'Étape':50s}   {'Lignes':>13s}   {'% du total brut':>15s}")
print("-" * 82)
for k, v in funnel_commercial.items():
    pct = v / funnel_commercial["1_total_lignes_brutes"] * 100
    print(f"{k:50s} : {v:>13,}   {pct:14.2f}%")

print()
print(f"5. Après géo + prix connu (lignes)          : {len(df_final_lignes_commercial):,}")
print(f"6. Après règle vente en bloc (transactions)  : {len(transactions_finales_commercial):,}")
print(f"7. Après seuil plausibilité €/m² (final)     : {len(transactions_vraiment_finales_commercial):,}")

Étape                                                       Lignes   % du total brut
----------------------------------------------------------------------------------
1_total_lignes_brutes                              :    20,382,915           100.00%
2_apres_dedoublonnage_strict                       :    18,859,504            92.53%
3_vente_ou_vefa_tout_type                          :    18,524,590            90.88%
4_perimetre_commercial                             :       625,797             3.07%

5. Après géo + prix connu (lignes)          : 601,520
6. Après règle vente en bloc (transactions)  : 391,232
7. Après seuil plausibilité €/m² (final)     : 391,188


### Interprétation — entonnoir

**Tableau complet** :

| Étape | Lignes/Transactions | % du brut |
|---|---|---|
| 1. Total brut | 20 382 915 | 100.00% |
| 2. Après dédoublonnage strict | 18 859 504 | 92.53% |
| 3. Vente + VEFA (tout type) | 18 524 590 | 90.88% |
| 4. Périmètre commercial/industriel | 625 797 | 3.07% |
| 5. Après géo + prix connu (lignes) | 601 520 | 2.95% |
| 6. Après règle vente en bloc (transactions) | 391 232 | 1.92% |
| 7. **Après seuil plausibilité €/m² (final)** | **391 188** | **1.92%** |

**Interprétation** : le périmètre commercial final représente **1.92%** du fichier brut — bien plus
restreint que le périmètre résidentiel (23.29%), cohérent avec le poids initial du type_local ciblé
(3.19% du brut avant tout filtre, section 2 du notebook 03). Le filtre le plus sélectif reste, comme
en résidentiel, le périmètre lui-même (§4, -87.81 points) ; toutes les étapes suivantes (géo/prix,
vente en bloc, plausibilité) affinent ensuite un volume déjà bien circonscrit.

La règle vente en bloc (§6) retire proportionnellement plus qu'en résidentiel à l'étape équivalente
— cohérent avec le taux de candidats vente-en-bloc plus élevé trouvé en section 11 (22.9% contre
14.7% en résidentiel).

**Ce tableau clôt l'EDA et la définition des règles pour le périmètre commercial/industriel.**

---

## Notes / prochaines étapes

### ✅ PHASE 1bis (commercial/industriel) — TERMINÉE

Toutes les règles de nettoyage sont définies, justifiées et appliquées de bout en bout :

- [x] Dédoublonnage strict (2.78%)
- [x] Filtre Vente + VEFA (VEFA à 3.43%, plus fréquent qu'en résidentiel)
- [x] Périmètre commercial/industriel (`type_local == 'Local industriel. commercial ou assimilé'`)
- [x] Vente en bloc : agrégation (≤9 lignes, 391 232 transactions) / exclusion (>9 lignes, 2 999
      transactions), avec correction du biais de surface dupliquée
- [x] Géographie (règle reprise du résidentiel, coordonnées valides, DOM-TOM distingués)
- [x] Seuil de plausibilité €/m² définitif : 1 100 000€/m² (44 transactions exclues, dont le cas
      Marseille à 14.15 milliards €)

### Volume final du périmètre commercial/industriel nettoyé

**391 188 transactions** — 1.92% du fichier brut (20 382 915 lignes). C'est le dataset d'entrée
définitif pour ce périmètre, à combiner avec les 4 748 082 transactions résidentielles (notebook 02)
pour la Phase 2 (Talend) et la Phase 4 (modèle ML).

### Limites assumées, à garder en tête
- Le seuil €/m² (1.1M€/m²) est une première estimation basée sur un percentile, pas un examen cas par
  cas exhaustif comme pour les seuils résidentiels
- La règle géographique n'a pas été revérifiée spécifiquement sur ce périmètre
- Le traitement des rares transactions multi-lignes à `valeur_fonciere` différente (non identique)
  reprend par défaut la même logique que le résidentiel, sans revérification
- `nombre_pieces_principales` reste exclu des règles de qualité pour ce périmètre
- 18 164 transactions (4.6%) sans surface exploitable ne bénéficient pas du contrôle de plausibilité

### Prochaines étapes (hors EDA)
- [ ] Committer/pousser les 2 fichiers finaux : `03_exploration_commercial.ipynb`,
      `04_regles_nettoyage_commercial.ipynb`
- [ ] **L'EDA (résidentiel + commercial) est maintenant terminée.** Passer à la Phase 2 : traduire
      les règles des deux périmètres en jobs Talend, charger le résultat dans PostgreSQL
      (`immo_france`, schéma en étoile)

### Robustesse
`keep_mask.npy` réutilisé tel quel depuis le résidentiel (dédoublonnage global, valable pour tout
périmètre) — pas besoin de `keep_mask_commercial` séparé.